In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("ReadJSON").getOrCreate()

In [15]:
from pathlib import Path

# Resolve absolute path to data/raw from the notebook location
notebook_dir = Path().resolve()  # notebooks/
data_path = notebook_dir.parent / "data" / "raw"

In [16]:
print(f"Current dir: {notebook_dir}")
print(f"Data path: {data_path}")
print(f"Path exists: {data_path.exists()}")
print(f"JSON files found: {list(data_path.glob('*.json'))}")

Current dir: /Users/snehamungre/projects/crypto_market_analysis/notebooks
Data path: /Users/snehamungre/projects/crypto_market_analysis/data/raw
Path exists: True
JSON files found: [PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-03_09-48-58.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-27_10-24-30.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-26_08-43-05.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-02_10-23-02.json')]


In [17]:
from pyspark.sql.types import *

schema = StructType(
    [
        StructField("ath", DoubleType(), True),
        StructField("ath_change_percentage", DoubleType(), True),
        StructField("ath_date", StringType(), True),
        StructField("atl", DoubleType(), True),
        StructField("atl_change_percentage", DoubleType(), True),
        StructField("atl_date", StringType(), True),
        StructField("circulating_supply", DoubleType(), True),
        StructField("current_price", DoubleType(), True),
        StructField("fully_diluted_valuation", LongType(), True),
        StructField("high_24h", DoubleType(), True),
        StructField("id", StringType(), True),
        StructField("image", StringType(), True),
        StructField("last_updated", StringType(), True),
        StructField("low_24h", DoubleType(), True),
        StructField("market_cap", LongType(), True),
        StructField("market_cap_change_24h", DoubleType(), True),
        StructField("market_cap_change_percentage_24h", DoubleType(), True),
        StructField("market_cap_rank", LongType(), True),
        StructField("max_supply", DoubleType(), True),
        StructField("name", StringType(), True),
        StructField("price_change_24h", DoubleType(), True),
        StructField("price_change_percentage_24h", DoubleType(), True),
        StructField(
            "roi",
            StructType(
                [
                    StructField("currency", StringType(), True),
                    StructField("percentage", DoubleType(), True),
                    StructField("times", DoubleType(), True),
                ]
            ),
            True,
        ),
        StructField(
            "sparkline_in_7d",
            StructType(
                [
                    StructField("price", ArrayType(DoubleType()), True),
                ]
            ),
            True,
        ),
        StructField("symbol", StringType(), True),
        StructField("total_supply", DoubleType(), True),
        StructField("total_volume", DoubleType(), True),
    ]
)

In [19]:
from pyspark.sql.functions import to_timestamp

df = spark.read.option("multiline", "true").schema(schema).json(str(data_path))
df = (
    df.withColumn("last_updated", to_timestamp("last_updated"))
    .withColumn("ath_date", to_timestamp("ath_date"))
    .withColumn("atl_date", to_timestamp("atl_date"))
    .drop("image", "symbol")
)

df.show(7)

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+-----------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+--------+--------------------+---------------------------+--------------------+--------------------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|         id|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|    name|    price_change_24h|price_change_percentage_24h|                 roi|     sparkline_in_7d|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+---------------------+------------

In [20]:
# Ranking by current price
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("id").orderBy(desc("last_updated"))

most_updated_data = (
    df
    .withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
)

In [21]:
ranking_df = most_updated_data[
    "name", "current_price", "market_cap", "last_updated", "total_volume"
]

ranking_df.show(4)

+--------+-------------+----------+--------------------+------------+
|    name|current_price|market_cap|        last_updated|total_volume|
+--------+-------------+----------+--------------------+------------+
|    A7A5|   0.01273556| 499149601|2026-03-03 09:48:...|     5585.21|
|    Aave|       119.44|1814877547|2026-03-03 09:48:...|5.63396201E8|
|Algorand|     0.086614| 768713218|2026-03-03 09:48:...| 3.5332216E7|
|   Aptos|     0.983245| 767386917|2026-03-03 09:48:...|1.40712028E8|
+--------+-------------+----------+--------------------+------------+
only showing top 4 rows


## Ranking current price

In [22]:
ranking_df.sort(desc("current_price")).show(10)

+------------+-------------+-------------+--------------------+---------------+
|        name|current_price|   market_cap|        last_updated|   total_volume|
+------------+-------------+-------------+--------------------+---------------+
|     Bitcoin|      67999.0|1361180043470|2026-03-03 09:48:...|5.9590593487E10|
|    PAX Gold|       5348.9|   2572967591|2026-03-03 09:48:...|   7.61452059E8|
| Tether Gold|      5299.67|   2993342849|2026-03-03 09:48:...|   9.52257823E8|
|    Ethereum|      1996.47| 241265092029|2026-03-03 09:48:...|2.7004116266E10|
|         BNB|       630.46|  86015000913|2026-03-03 09:48:...|  1.526222787E9|
|Bitcoin Cash|       441.51|   8833981598|2026-03-03 09:48:...|   3.51887499E8|
|      Monero|       337.43|   6228619583|2026-03-03 09:48:...|    9.9150255E7|
|       Zcash|       220.38|   3659559050|2026-03-03 09:48:...|   2.66846707E8|
|   Bittensor|        182.3|   1752867787|2026-03-03 09:48:...|   1.23888706E8|
|        Aave|       119.44|   181487754

## Ranking Market Cap

In [23]:
ranking_df.sort(desc("market_cap")).show(10)

+------------+-------------+-------------+--------------------+---------------+
|        name|current_price|   market_cap|        last_updated|   total_volume|
+------------+-------------+-------------+--------------------+---------------+
|     Bitcoin|      67999.0|1361180043470|2026-03-03 09:48:...|5.9590593487E10|
|    Ethereum|      1996.47| 241265092029|2026-03-03 09:48:...|2.7004116266E10|
|      Tether|      0.99988| 183606483628|2026-03-03 09:48:...|9.7552146231E10|
|         BNB|       630.46|  86015000913|2026-03-03 09:48:...|  1.526222787E9|
|         XRP|         1.37|  83576314263|2026-03-03 09:48:...|  3.540064814E9|
|        USDC|     0.999972|  75968433160|2026-03-03 09:48:...|1.8120496526E10|
|      Solana|        85.93|  48980490168|2026-03-03 09:48:...|  5.861852668E9|
|        TRON|     0.282376|  26762706445|2026-03-03 09:48:...|   4.27720063E8|
|Figure Heloc|        1.031|  15926715795|2026-03-03 09:48:...|    1.2233968E7|
|    Dogecoin|     0.091627|  1548392281

26/03/03 16:23:05 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 910168 ms exceeds timeout 120000 ms
26/03/03 16:23:05 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/03 16:23:12 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$